# 01 · NLLB-200 on Sindhi: evaluate & export for the app

Scores Meta's NLLB-200 checkpoints on **FLORES-200 devtest** (Sindhi ↔ English, chrF++ / BLEU),
then exports the chosen one to **CTranslate2 int8** — the format `src/translator.py` loads on a CPU.

**Output:** `/kaggle/working/nllb-ct2-int8.zip` → download from the *Output* tab, unzip into
`models/nllb-200-distilled-600M-ct2-int8/` (or set `NLLB_MODEL_DIR`), restart the app.

In [ ]:
# Session options (right panel): Accelerator = GPU T4 x2 (or P100), Internet = On.
!pip install -q -U transformers==4.57.1 accelerate sentencepiece sacrebleu ctranslate2==4.6.0 peft datasets
import torch, os
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# FLORES-200 devtest (1012 sentences) -- public download, no login needed
!wget -q -nc https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz && tar xzf flores200_dataset.tar.gz
def read(path):
    with open(path, encoding="utf-8") as f:
        return [l.rstrip("\n") for l in f]
flores = {"sd": read("flores200_dataset/devtest/snd_Arab.devtest"),
          "en": read("flores200_dataset/devtest/eng_Latn.devtest")}
CODES = {"sd": "snd_Arab", "en": "eng_Latn"}
print(len(flores["sd"]), flores["sd"][0][:80])

In [ ]:
import sacrebleu, time
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

def translate(model, tok, sentences, src, tgt, batch_size=32, beams=4):
    tok.src_lang = CODES[src]
    out = []
    for i in range(0, len(sentences), batch_size):
        batch = tok(sentences[i:i + batch_size], return_tensors="pt", padding=True, truncation=True, max_length=256).to(model.device)
        with torch.no_grad():
            gen = model.generate(**batch, forced_bos_token_id=tok.convert_tokens_to_ids(CODES[tgt]),
                                 num_beams=beams, max_new_tokens=256)
        out += tok.batch_decode(gen, skip_special_tokens=True)
    return out

def score(model, tok, n=1012):
    res = {}
    for src, tgt in (("sd", "en"), ("en", "sd")):
        t = time.time()
        hyps = translate(model, tok, flores[src][:n], src, tgt)
        refs = flores[tgt][:n]
        res[f"{src}-{tgt}"] = {"chrf++": round(sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score, 1),
                               "bleu": round(sacrebleu.corpus_bleu(hyps, [refs]).score, 1),
                               "sec": round(time.time() - t)}
    return res

In [ ]:
results = {}
for name in ["facebook/nllb-200-distilled-600M", "facebook/nllb-200-distilled-1.3B"]:
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForSeq2SeqLM.from_pretrained(name, torch_dtype=torch.float16).cuda().eval()
    results[name] = score(model, tok)
    print(name, results[name])
    del model; torch.cuda.empty_cache()

Pick the model to ship. The 600M one is what the app uses by default (≈0.6–3 s/sentence on a laptop CPU);
the 1.3B one is usually a few chrF++ points better but ~2× slower on CPU.

In [ ]:
CHOSEN = "facebook/nllb-200-distilled-600M"   # change to the 1.3B model if it's worth the speed cost
!ct2-transformers-converter --model {CHOSEN} --quantization int8 --output_dir nllb-ct2-int8 --force \
    --copy_files tokenizer.json tokenizer_config.json special_tokens_map.json sentencepiece.bpe.model
!cd /kaggle/working && zip -qr nllb-ct2-int8.zip nllb-ct2-int8 && ls -lh nllb-ct2-int8.zip
import json; json.dump(results, open("/kaggle/working/nllb_flores_results.json", "w"), indent=1)

In [ ]:
# sanity check: the exported model through the same code path as src/translator.py
import ctranslate2, sentencepiece
sp = sentencepiece.SentencePieceProcessor(model_file="nllb-ct2-int8/sentencepiece.bpe.model")
tr = ctranslate2.Translator("nllb-ct2-int8", device="cuda")
src = ["snd_Arab"] + sp.encode("سنڌ هڪ خوبصورت صوبو آهي.", out_type=str) + ["</s>"]
hyp = tr.translate_batch([src], target_prefix=[["eng_Latn"]])[0].hypotheses[0][1:]
print(sp.decode(hyp))